In [ ]:
# 실행 시간 측정: 단일 AEDTPLT export
import time

start_aedtplt = time.perf_counter()

# 첫 번째 variation만 추출
test_table_1_aedtplt = ParametricTable.iloc[:1]

print("=" * 70)
print(f"⏱️  AEDTPLT Export 시간 측정 (1개 Variation)")
print("=" * 70)

# AEDTPLT export 실행
aedtplt_files_test = export_aedtplt_for_variations(
    m2d_obj=m2d,
    parametric_table=test_table_1_aedtplt,
    quantity="Mag_B",
    solution="Setup1 : Transient",
    assignment="AllObjects",
    output_dir=r"D:\KDHe10\e10_example\AEDTPLT_Exports_Test",
    intrinsics={"Time": "0.06s"}
)

elapsed_aedtplt = time.perf_counter() - start_aedtplt

print(f"\n{'='*70}")
print("📊 AEDTPLT 성능 측정 결과")
print(f"{'='*70}")
print(f"✅ 총 소요 시간: {elapsed_aedtplt:.3f} 초")
print(f"📦 생성된 파일: {len(aedtplt_files_test)}개")
print(f"\n💡 전체 {len(ParametricTable)}개 variation 예상 시간:")
print(f"   약 {elapsed_aedtplt * len(ParametricTable):.1f} 초 ({elapsed_aedtplt * len(ParametricTable) / 60:.1f} 분)")
print(f"{'='*70}")

# Maxwell 2D Field Data Export
## Setup: AEDT Connection & Utilities
1. Mesh는 case로 내보내기
2. fld로 field내보내기
## 📚 패키지 정보

이 노트북은 `aedt_utils` 패키지를 사용합니다.

패키지 위치: `d:\KangDH\Emlab_emach\pyAEDT\aedt_utils\`

### 주요 기능:
- **Connection**: AEDT Desktop 자동 연결
- **Maxwell**: Maxwell 2D 디자인 자동 연결

### 사용 가능한 함수:
```python
from aedt_utils import (
    smartAedtConnector,      # 스마트 연결 (권장)
    quickConnect,            # 빠른 연결
    getDesktopConnection,    # Desktop 연결
    checkCurrentDesktopStatus, # 상태 확인
    getRunningMaxwell2d,     # Maxwell 2D 연결
)
```

import sys
sys.path.append(r'd:\KangDH\Emlab_emach\pyAEDT')

# AEDT Utils 패키지 import
from aedt_utils import (
    smartAedtConnector,
    quickConnect,
    getDesktopConnection,
    checkCurrentDesktopStatus,
    getAedtProcessesDetailed,
)

# 기본 설정
AEDT_VERSION = "2025.2"
NUM_CORES = 8
NG_MODE = False  # Open AEDT UI when it is launched.


## Load Maxwell 2D Model

In [ ]:
# e10 Model
AEDT_VERSION = "2025.2"
NUM_CORES = 8
NG_MODE = False  # Open AEDT UI when it is launched.
# aedt_file=r"D:\KDHe10\e10_example\e10_tutorial_ANSYSEM_2D.aedt"
# aedt_file=r"E:\KDH\e10\e10_DOE\e10_DOE.opd\AMOP\Design0001\e10_DOE.aedt"
# m2d = ansys.aedt.core.Maxwell2d(
#     project=aedt_file,
#     version=AEDT_VERSION,
#     new_desktop=False,
#     non_graphical=NG_MODE
# )
# Maxwell 2D 유틸리티 import
from aedt_utils import getRunningMaxwell2d

# 사용 예시
print("="*70)
print("🔌 현재 실행 중 Maxwell2d 객체 가져오기")
print("="*70)

m2d_running = getRunningMaxwell2d()

if m2d_running:
    print(f"Design Type: {m2d_running.design_type}")
    print(f"Variables: {list(m2d_running.variable_manager.variables.keys())[:5]} ...")
else:
    print("⚠️ Maxwell2d 객체를 가져오지 못했습니다.")

m2d=m2d_running



## 조각 코드

In [ ]:
orepSetupObj=m2d.oreportsetup
oDeskObj=m2d.odesktop
ProjObj=oDeskObj.GetProjects()
DesignListObj=ProjObj[0].GetDesigns()
DesignObj=ProjObj[0].GetDesigns()

ReporterObj=DesignObj[0].GetModule("ReportSetup")
# Sol

ReportNames=ReporterObj.GetAllReportNames()
ReportTypes=ReporterObj.GetAvailableReportTypes()
Solutions={}
Solutions[0]=ReporterObj.GetAvailableSolutions(ReportTypes[0])
Solutions[1]=ReporterObj.GetAvailableSolutions(ReportTypes[1])
Solutions[2]=ReporterObj.GetAvailableSolutions(ReportTypes[2])

print("GetAvailableReportTypes",ReportTypes)
# 
TransientDisp=ReporterObj.GetAvailableDisplayTypes(ReportTypes[0])
FieldDisp=ReporterObj.GetAvailableDisplayTypes(ReportTypes[1])
ContextList=ReporterObj.GetSolutionContexts(ReportTypes[0],TransientDisp[0],Solutions[0][0])
print("GetSolutionContexts ",ContextList)


print("GetAvailabe Solution",Solutions)
AllCatList=ReporterObj.GetAllCategories(ReportTypes[0],TransientDisp[0],Solutions[0][0],ContextList[0])
QuantList=ReporterObj.GetAllQuantities(ReportTypes[0],TransientDisp[0],Solutions[0][0],ContextList[0],AllCatList[0])
print("GetAllReportNames ",ReportNames)
print("GetAllCategories",AllCatList)
print("GetAllQuantities",QuantList)
print("ReporterObj, GetChildNames", ReporterObj.GetChildNames())
EMFObj=ReporterObj.GetChildObject(ReporterObj.GetChildNames()[0])
# print(EMFObj.GetChildNames())
Ph1Obj=EMFObj.GetChildObject(EMFObj.GetChildNames()[0])
print(Ph1Obj.GetChildNames())
# curve1Obj=Ph1Obj.GetChildObject(Ph1Obj.GetChildNames()[0])
# curve1Obj.GetDataModel()
variations = " ".join(
        f"{key}=\\'{value}\\'" for key, value in m2d.available_variations.nominal_w_values_dict.items()
    )
variations
m2dPost=m2d.post
m2dPost.available_display_types()
m2dPost.available_report_quantities()


In [ ]:
# SetupModuleObj.GetSetups
oProject = oDeskObj.GetActiveProject()

oDesign = oProject.GetActiveDesign()

# oDesign.GetModule("AnalysisSetup").GetSetups
SetupObj= oDesign.GetModule("AnalysisSetup")
VariableNamesList=oDesign.GetVariables()
VariableNamesList

oDesign.GetVariableValue('PhaseAdvance')
optimetricModuleObj=oDesign.GetModule("Optimetrics")
OptiType=optimetricModuleObj.GetChildTypes()
optimetricModuleObj.GetChildNames()
OptiObj=optimetricModuleObj.GetChildObject(optimetricModuleObj.GetChildNames()[0])
OptiObjPropNames=OptiObj.GetPropNames()
OptiObjPropNames
hasResult=OptiObj.HasResult()
oParamSetup = optimetricModuleObj.GetChildObject('ParametricSetup1')
for i in range(0, len(OptiObjPropNames)):
    print(optimetricModuleObj.GetPropValue("ParametricSetup1\\" + OptiObjPropNames[i]))
oModuleSol = oDesign.GetModule("Solutions")

# 추후에 이걸 이용하도록 파이썬 파일내 함수를 수정해줘
OptiObj.GetResults()    

SetupObj=AnalysisObj.GetChildObject(AnalysisObj.GetChildNames()[0])
SetupObj=SetupModuleObj.GetSetups()[0]

m2d.set_active_design(m2d.design_list[0])
orepSetupObj.GetSolutionContexts('Transient',"Rectangular Plot","Setup1:LastAdaptive")
m2dPost.get_solution_data_per_variation(solution_type='Transient',setup_sweep_name='ParametricSetup1',context=con)


## AEDT List 추출

E:\KDH\e10\e10_DOE\e10_DOE.opd\AMOP

In [ ]:
# AEDT 파일 관리 유틸리티 import
import pandas as pd
from aedt_file_utils import find_aedt_files, find_lock_files, remove_lock_files

# AEDT 파일 검색 예시
target_dir = r"E:\KDH\e10\e10_DOE\e10_DOE.opd\AMOP"
# target_dir= r"D:\KDHe10\e10_DOE\e10_DOE.opd\AMOP"
# target_dir=r"C:\e10_DOE\e10_DOE.opd\AMOP"

# 재귀 검색 깊이 설정 (옵션)
# max_depth=None   → 무제한 재귀 검색 (기본값)
# max_depth=0      → 현재 디렉토리만
# max_depth=1      → 1단계 하위 디렉토리까지
# max_depth=2      → 2단계 하위 디렉토리까지
MAX_DEPTH = 1  # 원하는 깊이로 변경 가능

aedt_files = find_aedt_files(target_dir, recursive=True, max_depth=MAX_DEPTH)
lock_files = find_lock_files(target_dir, recursive=True, max_depth=MAX_DEPTH)
result = remove_lock_files(lock_files)
# print(result['message'])
import ansys.aedt.core
from concurrent.futures import ProcessPoolExecutor, as_completed
import time

# 별도 파일에서 함수 import (ProcessPoolExecutor 호환)

from aedt_parallel_processor import process_aedt_file
# DataFrame으로 변환하여 표시
if aedt_files:
    df_aedt = pd.DataFrame(aedt_files)
# Lock 파일 찾기 및 삭제 예시
file_paths = df_aedt['full_path'].tolist()
file_paths

## 범용 파일 검색 (find_files)

`find_files` 함수를 사용하여 다양한 확장자와 패턴으로 파일을 검색할 수 있습니다.

### 사용 예시:
- CSV 파일 검색
- 특정 패턴이 포함된 파일 검색
- 정규식을 사용한 고급 검색

##  AEDT 해석

여러 AEDT 파일을 병렬로 열고 Excitation 설정 및 Parametric Sweep을 자동으로 적용합니다.

CSV 파일에서 전류와 각도 데이터를 읽어 Optimetrics 설정을 생성합니다.

- **전류**: 5 steps
- **각도**: 6 steps
- **총 Variations**: 30개 (5 × 6)

 전류와 각도 변수 추출 및 Sweep 범위 설정

 유틸리티 함수: 객체 속성 검색

`find()` 함수를 사용하여 객체의 속성이나 메서드를 쉽게 검색할 수 있습니다.

In [ ]:
125-150

In [ ]:
"""
강건한 AEDT 파일 일괄 처리 루프 (함수화 버전 사용)

Notebook에 있던 긴 처리 코드를 외부 모듈로 분리했습니다.
`process_parametric_projects` 함수를 호출하여 동일 기능 수행.
"""

from parametric_batch_processor import process_parametric_projects

# 파라미터 설정
SETUP_NAME = "ParametricSetup1"
ipeak_steps = 5
phase_steps = 6
RECREATE_INCOMPLETE = True  # 불완전 결과 재실행 여부
LIMIT = 2  # 테스트용 제한 (None이면 전체)
VERBOSE = True

print("=" * 80)
print("🚀 Parametric Batch Processing 시작")
print("=" * 80)
print(f"총 파일 수: {len(file_paths)} | 처리 제한: {LIMIT}")

results = process_parametric_projects(
    m2d_obj=m2d,
    file_paths=file_paths,
    setup_name=SETUP_NAME,
    ipeak_steps=ipeak_steps,
    phase_steps=phase_steps,
    recreate_incomplete=RECREATE_INCOMPLETE,
    limit=LIMIT,
    verbose=VERBOSE,
)

# 요약 출력
success_count = sum(r.get("is_complete") for r in results)
error_count = sum(1 for r in results if "error" in r)
print("\n" + "=" * 80)
print("📊 최종 요약")
print("=" * 80)
print(f"✅ 완료된 프로젝트: {success_count}/{len(results)}")
print(f"❌ 오류 발생 프로젝트: {error_count}")
print("=" * 80)


In [ ]:
# Parametric Sweep 결과 검증 (validate_and_display_csv 함수 사용)
from aedt_csv_validator import validate_and_display_csv

# Parametric Sweep 설정
# IPeak: 5 steps (10A ~ 650.53A)
# PhaseAdvance: 6 steps (0deg ~ 90deg)
ipeak_steps = 5
phase_steps = 6

# 검증 실행 (Maxwell2d 객체로부터 자동으로 경로 추출)
validation_result = validate_and_display_csv(
    m2d_obj=m2d,
    setup_name="ParametricSetup1",
    expected_ipeak_steps=ipeak_steps,
    expected_phase_steps=phase_steps,
    verbose=True
)

## FLD Export Function Definition

In [ ]:
m2dpost=m2d.post
all_objects = m2d.modeler.object_names

# Modelplotter=m2dpost.get_model_plotter_geometries(generate_mesh=True,get_objects_from_aedt=True)
### mesh export as *.case 
import os 
desktop = ansys.aedt.core.Desktop() 
pjtPath=desktop.project_path()
prjName=desktop.active_project().GetName()
filePath=os.path.join(pjtPath, prjName) 
pjt=desktop.load_project(filePath)
setup=pjt.get_setup(name='Setup1')
FieldReporter=pjt.get_module("FieldsReporter")

SetupObj=m2d.get_setup('Setup1')
{"Time":SetupObj.props['MaxTimeStep']}
import ansys.aedt.core.visualization.plot.pyvista as AEDTvista
AEDTvista
import ansys.aedt.core.visualization.post as AEDTpost
AEDTpost

## plot Field

In [ ]:
# ===== 방법 1: ReportSetup 모듈을 통한 직접 데이터 추출 =====

# Reporter 모듈 가져오기
oDesign = m2d.odesign
oReporter = oDesign.GetModule("ReportSetup")

# 솔루션 정보 설정
report_type = "Transient"  # 또는 "Fields"
display_type = "Rectangular Plot"  # 또는 "Data Table"
solution = "Setup1 : Transient"  # 솔루션 이름


In [ ]:

# Context 정보 가져오기 (어떤 variation, time point 등)
contexts = oReporter.GetSolutionContexts(report_type, display_type, solution)
print(f"Available Contexts: {contexts}")

# 특정 context 선택 (예: 첫 번째)
if contexts:
    context = contexts[0]
    print(f"Selected Context: {context}")
    
    # 사용 가능한 카테고리와 수량 확인
    categories = oReporter.GetAllCategories(report_type, display_type, solution, context)
    print(f"Categories: {categories}")
    
    if categories:
        quantities = oReporter.GetAllQuantities(report_type, display_type, solution, context, categories[0])
        print(f"Quantities: {quantities[:10]}")  # 처음 10개만 출력

In [ ]:
# ===== 방법 2: GetSolutionDataPerVariation을 직접 호출 (안전 버전) =====

# 이 메서드는 pyAEDT의 SolutionData를 생성할 때 내부적으로 사용하는 것과 동일합니다
oReporter = m2d.odesign.GetModule("ReportSetup")

# 매개변수 설정
report_type = "Transient"
solution_name = "Setup1 : Transient"
context = ""  # 빈 문자열

# families 매개변수 수정: 리스트가 아닌 개별 요소로 전달
# 데이터 추출 (이것이 pyAEDT 내부에서 사용되는 원시 데이터)
try:
    raw_solution_data = oReporter.GetSolutionDataPerVariation(
        report_type,
        solution_name,
        context,
        ["Time:=", ["All"]],     # Sweep 변수 설정
        ["Moving1.Torque"]       # 원하는 수량 (expression)
    )
    
    print(f"✅ 데이터 추출 성공")
    print(f"Raw Solution Data Type: {type(raw_solution_data)}")
    print(f"Number of variations: {len(raw_solution_data)}")
    
    # 첫 번째 variation 확인
    if len(raw_solution_data) > 0:
        first_var = raw_solution_data[0]
        print(f"First Variation Type: {type(first_var)}")
        
        # variation의 메서드 확인
        print(f"\nAvailable methods (일부):")
        methods = [m for m in dir(first_var) if not m.startswith('_')]
        print(f"  {methods[:10]}")
        
except Exception as e:
    print(f"❌ Error getting solution data: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
        variations = h3d_potter_horn.available_variations.get_independent_nominal_values()


In [ ]:
SolutionData(first_var)

In [ ]:
# ===== 방법 3: pyAEDT 고수준 API 사용 (권장) =====

# SolutionData를 직접 생성하지 말고 pyAEDT의 post 모듈 사용
# 이 방법이 가장 안전하고 안정적입니다

try:
    # m2d.post.get_solution_data() 사용
    solution_data_obj = m2d.post.get_solution_data(
        expressions=["Moving1.Torque", "Moving1.Speed"],
        setup_sweep_name="Setup1 : Transient",
        domain="Time"
    )
    
    if solution_data_obj:
        print(f"✅ SolutionData 객체 생성 완료")
        print(f"Expressions: {solution_data_obj.expressions}")
        print(f"Primary Sweep: {solution_data_obj.primary_sweep}")
        print(f"Number of Variations: {solution_data_obj.number_of_variations}")
        
        # Intrinsics 확인 (Sweep 변수)
        if solution_data_obj.intrinsics:
            print(f"Intrinsics: {list(solution_data_obj.intrinsics.keys())}")
        
        # 데이터 접근 예시
        torque_data = solution_data_obj.data_real('Moving1.Torque')
        time_values = solution_data_obj.primary_sweep_values
        
        print(f"\nTorque Data (첫 5개): {torque_data[:5]}")
        print(f"Time Values (첫 5개): {time_values[:5]}")
        print(f"Total data points: {len(torque_data)}")
    else:
        print("❌ 데이터를 가져오지 못했습니다.")
        
except Exception as e:
    print(f"❌ Error: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
# ===== 방법 4: Parametric Sweep 결과에 대한 SolutionData (안전 버전) =====

# pyAEDT 고수준 API를 사용한 Parametric Sweep 데이터 추출
try:
    # get_solution_data_per_variation 사용
    solution_data_list = m2d.post.get_solution_data_per_variation(
        setup_sweep_name="ParametricSetup1",
        expressions=["Moving1.Torque"],
        domain="Time"
    )
    
    if solution_data_list:
        print(f"✅ Parametric SolutionData 생성 완료")
        print(f"Number of variations: {len(solution_data_list)}개")
        
        # 첫 번째 variation 데이터 확인
        if len(solution_data_list) > 0:
            first_solution = solution_data_list[0]
            
            print(f"\n첫 번째 Variation 정보:")
            print(f"  Expressions: {first_solution.expressions}")
            print(f"  Primary Sweep: {first_solution.primary_sweep}")
            
            if first_solution.intrinsics:
                print(f"  Intrinsics: {list(first_solution.intrinsics.keys())}")
            
            # 데이터 추출
            torque_data = first_solution.data_real('Moving1.Torque')
            time_values = first_solution.primary_sweep_values
            
            print(f"\n  Torque Data (첫 5개): {torque_data[:5]}")
            print(f"  Time Values (첫 5개): {time_values[:5]}")
            print(f"  Total points: {len(torque_data)}")
            
        # 모든 variation 순회 예시
        print(f"\n모든 Variation 순회:")
        for idx, sol_data in enumerate(solution_data_list[:3]):  # 처음 3개만
            print(f"  Variation {idx}: {sol_data.variations}")
    else:
        print("❌ Parametric 데이터를 가져오지 못했습니다.")
        print("   Parametric Setup이 실행되었는지 확인하세요.")
        
except Exception as e:
    print(f"❌ Error: {e}")
    import traceback
    traceback.print_exc()

### ⚠️ 중요 참고사항

**방법 2 (GetSolutionDataPerVariation 직접 호출)**
- 원시 COM 객체를 직접 다루므로 조심스럽게 사용해야 함
- `families` 매개변수 형식이 정확해야 함: `["변수명:=", ["값1", "값2", ...]]`
- 에러가 발생하면 AEDT가 불안정해질 수 있음

**방법 3 (pyAEDT 고수준 API - 권장)**
- `m2d.post.get_solution_data()` 사용
- 가장 안전하고 안정적인 방법
- pyAEDT가 내부적으로 에러 처리를 해줌
- **이 방법을 우선 사용하는 것을 강력히 권장**

**방법 4 (Parametric Sweep - 고수준 API)**
- `m2d.post.get_solution_data_per_variation()` 사용
- Parametric Setup이 먼저 실행되어야 함
- 각 variation에 대해 별도의 SolutionData 객체 반환

**안전한 사용 순서:**
1. 먼저 방법 3 (고수준 API)을 시도
2. 특수한 경우에만 방법 2 (저수준 API) 사용
3. Parametric 결과는 방법 4 사용

### 핵심 요약

**SolutionData 객체를 수동으로 만드는 단계:**

1. **AEDT Reporter 모듈 가져오기**
   ```python
   oReporter = m2d.odesign.GetModule("ReportSetup")
   ```

2. **GetSolutionDataPerVariation() 호출**
   ```python
   raw_data = oReporter.GetSolutionDataPerVariation(
       report_type,      # "Transient", "Fields" 등
       solution_name,    # "Setup1 : Transient"
       context,          # "" 또는 특정 context
       families,         # Sweep 변수: ["Time:=", ["All"]]
       expressions       # ["Moving1.Torque", "Moving1.Speed"]
   )
   ```

3. **SolutionData 객체 생성**
   ```python
   from ansys.aedt.core.visualization.post.solution_data import SolutionData
   solution_data_obj = SolutionData(raw_data)
   ```

4. **데이터 접근**
   - `solution_data_obj.data_real(expression)` - 실수 데이터
   - `solution_data_obj.data_imag(expression)` - 허수 데이터
   - `solution_data_obj.data_magnitude(expression)` - 크기
   - `solution_data_obj.primary_sweep_values` - X축 값 (Time 등)
   - `solution_data_obj.variations` - 모든 variation 목록
   - `solution_data_obj.intrinsics` - Sweep 변수들

**주의사항:**
- `families` 매개변수 형식: `["변수명:=", ["값1", "값2", ...]]` 또는 `["All"]`
- Parametric Sweep의 경우: `"ParametricSetup1 : Setup1"` 형식 사용
- pyAEDT의 `m2d.post.get_solution_data()` 메서드도 내부적으로 동일한 방식 사용

## [Ongoing ] 테스트: 단일 파일 Export

전체 파일 처리 전에 단일 파일로 테스트합니다.

In [ ]:
from export_aedtplt import get_time_steps

# 현재 열린 AEDT 파일 확인
print("=" * 80)
print("🧪 export_aedtplt 단일 함수 테스트")
print("=" * 80)
print(f"📂 현재 프로젝트: {m2d.project_name}")
print(f"📁 프로젝트 경로: {m2d.project_path}")
print("=" * 80)

# Export 설정
OUTPUT_DIR = r"E:\KDH\e10\e10_DOE\e10_DOE.opd\AMOP\Design0125\FLD_Manual_Test"
SETUP_NAME = "Setup1"
PARAMETRIC_SETUP_NAME = "ParametricSetup1"
QUANTITY = "A_Vector"
PLOT_NAME = "A_Vector1"

print(f"\n📋 Export 설정:")
print(f"  출력 디렉토리: {OUTPUT_DIR}")
print(f"  Setup: {SETUP_NAME}")
print(f"  Parametric Setup: {PARAMETRIC_SETUP_NAME}")
print(f"  Quantity: {QUANTITY}")
print(f"  Plot Name: {PLOT_NAME}")
print("=" * 80)

In [ ]:
param=m2d.parametrics
oModule = param.optimodule
existing_setups = oModule.GetChildNames()
print(  existing_setups   )
m2d_obj=m2d
output_dir=OUTPUT_DIR
setup_name=SETUP_NAME
parametric_setup_name=PARAMETRIC_SETUP_NAME
quantity=QUANTITY
plot_name=PLOT_NAME
create_plot_if_missing=True
time_steps = get_time_steps(m2d)
len(time_steps)
m2d_obj.post.available_report_quantities()

# call export A field

In [ ]:
from export_aedtplt import export_aedtplt
res = export_aedtplt(
    m2d_obj,
    output_dir="D:/ExportPlots",
    variation_index=2  # 첫 번째 Variation
)

# export aedt debug

In [ ]:
import importlib
import export_aedtplt
importlib.reload(export_aedtplt)
from export_aedtplt import export_field
print('export_aedtplt reloaded; export_field ->', export_field)


## dg single Step export

In [ ]:
from export_aedtplt import *
DesignDir=r"E:\\KDH\\e10\\e10_DOE\\e10_DOE.opd\\AMOP\\Design0124\\"
file_path=DesignDir
output_dir=DesignDir
export_dir = Path(output_dir)
parametric_table=get_parametric_sweep_table(m2d)
ipeak_val=parametric_table.IPeak[0]
phase_val=parametric_table.PhaseAdvance[0]
time_steps=get_time_steps(m2d)
aedt_stem = Path(file_path).stem
oDesign = m2d.odesign
m2d_obj=m2d
oModule = oDesign.GetModule("FieldsReporter")

existing_plots = oModule.GetFieldPlotNames()
print(existing_plots)
all_object_names = m2d_obj.modeler.object_names
num_objects = len(all_object_names)
print(num_objects)
# PlotGeomInfo 구성
plot_geom_info = [1, "Surface", "FacesList", num_objects] + all_object_names
print(plot_geom_info)
plot_name='A_Vector1'
print(plot_name)

# oModule.SetPlotsViewSolutionContext(["A_Vector1"], "Setup1 : Transient", "Time="+time_steps[10])

# timeIndex=1
# oModule.ExportFieldPlot("A_Vector1", False, DesignDir+"vecA_Time"+str(timeIndex+1)+".aedtplt")



## dev

In [ ]:
time_idx=1
file_name_export = (
    f"{aedt_stem}_"
    f"IPeak{ipeak_val}_"
    f"Phase{phase_val}_"
    f"Time{time_idx:03d}.aedtplt"
)
print(file_name_export)
file_path_export = export_dir / file_name_export
print(file_path_export)
plot_name='A_vector1'
setup_name='Setup1'


# Hori san Code

In [ ]:
m2d.load_project(file_paths[124])

In [ ]:
import os, time
# from System.IO import File
# import ScriptEnv

oDesign = m2d.odesign
m2d_obj=m2d
oModule = oDesign.GetModule("FieldsReporter")
# sWorkingFolder=r"D:\KDHe10\e10_example"
#sWorkingFolder = r'D:\SingleFieldExportFromMaxwell\_2025R2_Test\Case2'
sTargetObjName =  'Stator_Lamination_Primitive'
sFieldQuantity = 'A'    #B, E, D, or H
# sWorkingFolder = r'D:\SingleFieldExportFromMaxwell\_2025R2_Test\Case1_2025R1'
sWorkingFolder = r'E:\KDH\e10\e10_DOE\e10_DOE.opd\AMOP\Design0125'
# sTargetObjName =  'OL_HopperAir'
# sFieldQuantity = 'B'    #B, E, D, or H
sSetupName = 'Setup1'
sVersion = ''           # empty means current AEDT ver.
#sVersion = '2024.1'    # Enforce to sexport 2024R1 format
aComp = ["X", "Y", "Z"]


In [ ]:
sWorkingFolder
sSetupName

In [ ]:
os.path.join(sWorkingFolder, sFieldQuantity + 'vec.fld')
from export_aedtplt import *
parametric_table=get_parametric_sweep_table(m2d)
parametric_table.head()

In [ ]:
oModule.EnterQty(sFieldQuantity)
oModule.CalcOp("Smooth")
oModule.EnterVol(sTargetObjName)
oModule.CalcOp("Value")
oModule.CalculatorWrite(
    os.path.join(sWorkingFolder, sFieldQuantity + 'vec.fld'), 
    ["Solution:=", sSetupName + " : Transient"], 
    [
        "Time:=", "0s",
        "IPeak:=", str(parametric_table.IPeak[0]),
        "PhaseAdvance:=", str(parametric_table.PhaseAdvance[0])
    ]
)

# final

In [ ]:
from export_aedtplt import *
parametric_table = get_parametric_sweep_table(m2d, 'ParametricSetup1')


In [ ]:
oDesign = m2d.odesign
oModule = oDesign.GetModule("FieldsReporter")


In [ ]:
# Reload module and run a quick single-variation export test (1 variation)
import importlib
import export_aedtplt
importlib.reload(export_aedtplt)
from export_aedtplt import export_field

print('export_field reloaded ->', export_field)

# quick test: single variation (0), all its time steps but small batch
out = export_field(
    m2d_obj=m2d,
    output_dir=Path(m2d.project_path) / 'FLD_Diagnostic_Test',
    setup_name='Setup1',
    parametric_setup_name='ParametricSetup1',
    field_quantity='A',
    target_object='Stator_Lamination_Primitive',
    variation_index=0,
    batch_size=10,
    save_wait_time=1,
    export_delay=0.05
)
print('Export test result:', out)


In [ ]:
from pathlib import Path

# 모든 variation과 time step
# 현재 AEDT 파일이 있는 디렉토리에 FLD_Export_All 폴더 생성
output_dir = Path(m2d.project_path) / "FLD_Export_All"
print(output_dir)

# 단일 variation, 단일 time step
res = export_field(
    m2d_obj=m2d,
    output_dir=output_dir,
    field_quantity="A",  # 또는 "B", "E", "D", "H"
    target_object="Stator_Lamination_Primitive",
    variation_index=1,  # 첫 번째 variation
)


### 💡 수동 테스트 가이드

**셀 실행 순서:**
1. **준비**: 변수 설정 셀 실행
2. **1단계**: 데이터 준비 (parametric_table, time_steps)
3. **2단계**: AEDT 모듈 준비 (oDesign, oModule)
4. **3단계**: 단일 Time Step 테스트 (각 API 호출을 개별 출력)
5. **4단계**: 여러 Time Steps Loop 테스트

**디버깅 포인트:**
- 3단계에서 어느 API 호출에서 팅기는지 확인
- 4단계에서 몇 번째 time step에서 팅기는지 확인
- `CalcStack("clear")` 호출 후에도 팅기는지 확인

**조정 가능한 변수:**
- `TEST_VARIATION_IDX`: 테스트할 variation (0부터 시작)
- `TEST_TIME_RANGE_END`: 테스트할 time step 개수 (5 → 10 → 20 → all)

In [ ]:
parametric_setup_name='ParametricSetup1'
setup_name='Setup1'

In [ ]:
# ===== 1단계: 데이터 준비 =====

# Parametric Table 가져오기
from  export_aedtplt import *
parametric_table = get_parametric_sweep_table(m2d, parametric_setup_name)
print(f"📊 Total Variations: {len(parametric_table)}")

# Time Steps 가져오기
time_steps = get_time_steps(m2d)
print(f"⏱️  Total Time Steps: {len(time_steps)}")


In [ ]:
TEST_VARIATION_IDX=0
TEST_TIME_RANGE_START=0
TEST_TIME_RANGE_END=15

In [ ]:

# 테스트할 Variation 선택
test_row = parametric_table.iloc[TEST_VARIATION_IDX]
ipeak_val = test_row['IPeak']
phase_val = test_row['PhaseAdvance']
print(f"\n🎯 Selected Variation {TEST_VARIATION_IDX}:")
print(f"  IPeak: {ipeak_val}")
print(f"  PhaseAdvance: {phase_val}")

# 테스트할 Time Steps 선택
test_time_steps = time_steps[TEST_TIME_RANGE_START:TEST_TIME_RANGE_END]
print(f"\n⏱️  Test Time Steps: {test_time_steps}")

In [ ]:
# ===== 준비: 변수 설정 (모든 Variation 순회 모드) =====
from export_aedtplt import *
from pathlib import Path

# 기본 설정
setup_name = "Setup1"
parametric_setup_name = "ParametricSetup1"

# Parametric table 준비 (없으면 생성)
parametric_table = get_parametric_sweep_table(m2d, parametric_setup_name)
if parametric_table is None:
    raise RuntimeError('parametric_table을 가져올 수 없습니다. ParametricSetup을 확인하세요.')

# 출력 디렉토리
output_dir_manual = Path(m2d.project_path) / "FLD_Manual_Test"
output_dir_manual.mkdir(parents=True, exist_ok=True)
output_dir = output_dir_manual

# 필드/대상 설정
field_quantity = "A"
target_object = "AllObjects"

# Time range (기본: 처음 5개까지 테스트)
TEST_TIME_RANGE_START = 0
TEST_TIME_RANGE_END = 5  # 처음 N개 time steps

# 모든 variation을 대상으로 동작하도록 인덱스 리스트 생성
NUM_VARIATIONS = len(parametric_table)
TEST_VARIATION_INDICES = list(range(NUM_VARIATIONS))

print(f"✅ 설정 완료: {NUM_VARIATIONS}개 variation을 처리합니다.")
print(f"  Output dir: {output_dir}")
print(f"  Time range (start..end-1): {TEST_TIME_RANGE_START}..{TEST_TIME_RANGE_END-1}")
print(f"  Variation indices: {TEST_VARIATION_INDICES}")

# 미리보기: 첫 variation 정보
if NUM_VARIATIONS > 0:
    ipeak_val, phase_val = parametric_table.iloc[0][['IPeak','PhaseAdvance']]
    print(f"  첫 variation 예시: idx=0, IPeak={ipeak_val}, PhaseAdvance={phase_val}")


In [ ]:
# ===== 소규모 직접 Export (중첩 for문, export_field 사용 안함) =====
# 구성: NVAR (variation 수), NTIME (time step 수)로 작게 테스트합니다.
from pathlib import Path
import time

# 사용자 설정 (작게 시작)
NVAR = min(3, len(parametric_table))  # 처음 3 variations
NTIME = min(5, len(time_steps))       # 처음 5 time steps
BATCH_SIZE = 10
SAVE_WAIT_TIME = 2
EXPORT_DELAY = 0.1

output_dir = Path(m2d.project_path) / 'FLD_Small_Test'
output_dir.mkdir(parents=True, exist_ok=True)

oDesign = m2d.odesign
oModule = oDesign.GetModule('FieldsReporter')

exported = 0
errors = []
start = time.perf_counter()

aedt_stem = Path(m2d.project_path).stem

print(f"소규모 Export 시작: NVAR={NVAR}, NTIME={NTIME}, output={output_dir}")

for vi in range(NVAR):
    row = parametric_table.iloc[vi]
    ipeak_val = row['IPeak']
    phase_val = row['PhaseAdvance']
    print(f"\n== Variation {vi} : IPeak={ipeak_val}, Phase={phase_val} ==")

    # (선택) 변수 설정: ChangeProperty로 로컬 변수 업데이트
    try:
        oDesign.ChangeProperty([
            "NAME:AllTabs",
            [
                "NAME:LocalVariableTab",
                ["NAME:PropServers", "LocalVariables"],
                [
                    "NAME:ChangedProps",
                    ["NAME:IPeak", "Value:=", str(ipeak_val)],
                    ["NAME:PhaseAdvance", "Value:=", str(phase_val)]
                ]
            ]
        ])
    except Exception as e:
        print(f"변수 설정 실패: {e}")

    for ti in range(NTIME):
        time_value = time_steps[ti]
        fname = f"{aedt_stem}_IPeak{ipeak_val}_Phase{phase_val}_Time{ti:03d}.fld"
        fpath = output_dir / fname

        try:
            oModule.CalcStack('clear')
            oModule.EnterQty(field_quantity)
            oModule.CalcOp('Smooth')
            oModule.EnterVol(target_object)
            oModule.CalcOp('Value')

            # flat list 인자 (안정적으로 동작하는 형태)
            args = [
                'Time:=', time_value,
                'IPeak:=', str(ipeak_val),
                'PhaseAdvance:=', str(phase_val)
            ]

            oModule.CalculatorWrite(str(fpath), ["Solution:=", f"{setup_name} : Transient"], args)
            oModule.CalcStack('clear')

            # 확인
            ok = fpath.exists() and fpath.stat().st_size > 0
            if ok:
                exported += 1
                print(f"[{vi},{ti}] OK -> {fname}")
            else:
                raise RuntimeError('파일 생성 실패')

            time.sleep(EXPORT_DELAY)

            if BATCH_SIZE > 0 and exported % BATCH_SIZE == 0:
                print(f"  배치 저장 (exported={exported}) -> saving...", end='')
                try:
                    oDesign.Save()
                except Exception as e:
                    print(f" save 실패: {e}")
                time.sleep(SAVE_WAIT_TIME)
                print(' done')

        except Exception as e:
            print(f"[{vi},{ti}] ERROR: {e}")
            errors.append({'variation': vi, 'time_idx': ti, 'error': str(e)})
            try:
                oModule.CalcStack('clear')
            except Exception:
                pass
            time.sleep(1)

elapsed = time.perf_counter() - start
print(f"\n완료: exported={exported}, errors={len(errors)}, time={elapsed:.1f}s")
if errors:
    print('첫 5 에러:', errors[:5])


In [ ]:
# ===== 2단계: AEDT 모듈 준비 =====

# Design과 FieldsReporter 모듈 가져오기
oDesign = m2d.odesign
oModule = oDesign.GetModule("FieldsReporter")

print(f"✅ oDesign: {type(oDesign)}")
print(f"✅ oModule: {type(oModule)}")

# 프로젝트 파일명
aedt_stem = Path(m2d.project_path).stem
print(f"📁 Project Name: {aedt_stem}")

## 🔧 수동 Export: export_field 내부 코드를 한 줄씩 실행

`export_field` 함수가 팅기는 원인을 찾기 위해 내부 로직을 꺼내서 단계별로 실행합니다.

In [ ]:
output_dir_manual

time_steps=get_time_steps(m2d)
# ===== 해결책 2: 더 작은 배치 + 긴 대기 시간 =====
import time

export_count = 0
error_count = 0
BATCH_SIZE = 15  # 15개로 감소
SAVE_WAIT_TIME = 5  # 5초 대기
EXPORT_DELAY = 0.2  # 각 export 사이 0.2초 대기

print(f"🔄 안전 배치 처리 (배치: {BATCH_SIZE}, 대기: {SAVE_WAIT_TIME}초)")
print(f"  총 Time Steps: {len(time_steps)}\n")

for idx, time_value in enumerate(time_steps):
    print(f"[{idx+1}/{len(time_steps)}] Time {idx}: {time_value} ", end="", flush=True)
    
    try:
        # 파일명 생성
        file_name = (
            f"{aedt_stem}_"
            f"IPeak{ipeak_val}_"
            f"Phase{phase_val}_"
            f"Time{idx:03d}.fld"
        )
        
        file_path = output_dir_manual / file_name
        
        # Calculator 스택 설정 및 Export
        oModule.EnterQty(field_quantity)
        oModule.CalcOp("Smooth")
        oModule.EnterVol(target_object)
        oModule.CalcOp("Value")
        
        oModule.CalculatorWrite(
            str(file_path),
            ["Solution:=", f"{setup_name} : Transient"],
            [
                "Time:=", time_value,
                "IPeak:=", str(ipeak_val),
                "PhaseAdvance:=", str(phase_val)
            ]
        )
        
        # 스택 정리
        oModule.CalcStack("clear")
        # oModule.ReleaseData()


        export_count += 1
        print("✅", end="", flush=True)
        
        # 각 export 후 짧은 대기
        time.sleep(EXPORT_DELAY)
        
        # 배치마다 Design 저장 + 긴 대기
        if (idx + 1) % BATCH_SIZE == 0:
            batch_num = (idx + 1) // BATCH_SIZE
            print(f" 💾 [배치 {batch_num}] 저장+대기 {SAVE_WAIT_TIME}초...", end="", flush=True)
            # oDesign.Save()
            time.sleep(SAVE_WAIT_TIME)
            print(" ✅", flush=True)
        else:
            print("", flush=True)
        
    except Exception as e:
        error_count += 1
        print(f"❌ {e}")
        try:
            oModule.CalcStack("clear")
        except:
            pass
        # 에러 후에도 대기
        time.sleep(1)

# 마지막 배치 저장
if export_count % BATCH_SIZE != 0:
    print(f"\n💾 최종 저장 중...", end="", flush=True)
    # oDesign.Save()
    time.sleep(SAVE_WAIT_TIME)
    print(" ✅")

print(f"\n{'='*60}")
print(f"📊 결과: ✅ {export_count} / ❌ {error_count}")
print(f"{'='*60}")

In [ ]:
# ===== Batch 변환: 모든 Variation (30) × 모든 TimeStep (45) =====
from pathlib import Path
import time
import csv

# 출력 폴더(FLD 파일들이 모여있는 폴더)를 지정하세요.
# 이미 사용중인 변수(output_dir_manual)가 있다면 그것을 사용합니다.
fld_dir = Path(r"E:/KDH/e10/e10_DOE/e10_DOE.opd/AMOP/Design0121/FLD_Manual_Test")
if 'output_dir_manual' in globals():
    fld_dir = Path(output_dir_manual)

print(f"FLD 디렉토리: {fld_dir}")
if not fld_dir.exists():
    raise FileNotFoundError(f"폴더가 없습니다: {fld_dir}")

# Parametric table과 time_steps는 노트북에 이미 존재해야 합니다.
if 'parametric_table' not in globals():
    raise RuntimeError('parametric_table 변수가 노트북에 없습니다. 먼저 1단계 셀을 실행하세요.')
if 'time_steps' not in globals():
    raise RuntimeError('time_steps 변수가 노트북에 없습니다. 먼저 1단계 셀을 실행하세요.')

# 변환 함수(이미 정의되어 있으면 재사용)
if 'convert_fld_to_h5' not in globals():
    import os, re, h5py, numpy as np
    FLOAT_RE = re.compile(r"[-+]?(?:\\d*\\.\\d+|\\d+\\.?(?:\\d*)?)(?:[eE][-+]?\\d+)?")
    def convert_fld_to_h5(fld_path, remove_fld=True, compression='gzip', compression_opts=6):
        fld_path = Path(fld_path)
        if not fld_path.exists():
            raise FileNotFoundError(f".fld file not found: {fld_path}")
        h5_path = fld_path.with_suffix('.h5')
        data_list = []
        with fld_path.open('r', encoding='utf-8', errors='ignore') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                parts = line.split()
                nums = None
                if len(parts) >= 4:
                    try:
                        cand = parts[:4]
                        nums = [float(x) for x in cand]
                    except Exception:
                        nums = None
                if nums is None:
                    found = FLOAT_RE.findall(line)
                    if len(found) >= 4:
                        try:
                            nums = [float(x) for x in found[:4]]
                        except Exception:
                            nums = None
                if nums is not None:
                    data_list.append(nums)
        if len(data_list) == 0:
            raise ValueError('No numeric data parsed from .fld file.')
        arr = np.asarray(data_list, dtype=np.float64)
        with h5py.File(h5_path, 'w') as hf:
            ds_kwargs = {}
            if compression:
                ds_kwargs['compression'] = compression
                ds_kwargs['compression_opts'] = compression_opts
            hf.create_dataset('field', data=arr, **ds_kwargs)
            hf.attrs['source_file'] = str(fld_path.name)
            hf.attrs['n_points'] = arr.shape[0]
        fld_size = fld_path.stat().st_size
        h5_size = h5_path.stat().st_size
        removed = False
        if remove_fld and h5_path.exists() and h5_size > 0:
            try:
                os.remove(fld_path)
                removed = True
            except Exception as e:
                print(f"⚠️ .fld 삭제 실패: {e}")
        return {'fld': str(fld_path), 'h5': str(h5_path), 'n_points': int(arr.shape[0]), 'fld_size': int(fld_size), 'h5_size': int(h5_size), 'removed': bool(removed)}

# 파일명 패턴 생성에 필요한 값들
# aedt_stem (프로젝트명) 사용
if 'aedt_stem' not in globals():
    # fallback: 프로젝트 경로에서 추출
    if 'm2d' in globals():
        aedt_stem = Path(m2d.project_path).stem
    else:
        raise RuntimeError('aedt_stem이나 m2d 변수가 없습니다. 프로젝트 스템을 정의해주세요.')

# 진행 파라미터
batch_sleep = 0.5  # batch 저장시 여유(초) — 필요한 경우 조정
per_file_sleep = 0.05
compression_level = 9

# 결과 저장
results = []
errors = []
start_time = time.perf_counter()

# parametric_table에서 IPeak/Phase값을 순회
for v_idx, row in parametric_table.iterrows():
    ipeak_val = row['IPeak']
    phase_val = row['PhaseAdvance']
    print(f"\n== Variation {v_idx+1}/{len(parametric_table)}: IPeak={ipeak_val}, Phase={phase_val} ==")
    for t_idx, time_value in enumerate(time_steps):
        # 파일명 규칙: {aedt_stem}_IPeak{ipeak_val}_Phase{phase_val}_Time{time_idx:03d}.fld
        fname = f"{aedt_stem}_IPeak{ipeak_val}_Phase{phase_val}_Time{t_idx:03d}.fld"
        fld_path = fld_dir / fname
        if not fld_path.exists():
            msg = f"Missing: {fld_path.name}"
            print(f" - {msg}")
            errors.append({'variation': v_idx, 'time_idx': t_idx, 'file': str(fld_path), 'error': 'missing'})
            continue
        try:
            res = convert_fld_to_h5(fld_path, remove_fld=True, compression='gzip', compression_opts=compression_level)
            results.append({'variation': v_idx, 'time_idx': t_idx, **res})
            print(f" - [{t_idx+1}/{len(time_steps)}] OK: {res['n_points']} pts, {res['h5_size']/1024/1024:.2f} MB")
            time.sleep(per_file_sleep)
        except Exception as e:
            print(f" - [{t_idx+1}/{len(time_steps)}] FAIL: {e}")
            import traceback
            traceback.print_exc()
            errors.append({'variation': v_idx, 'time_idx': t_idx, 'file': str(fld_path), 'error': str(e)})
    # 배치 여유
    time.sleep(batch_sleep)

elapsed = time.perf_counter() - start_time
print(f"\n완료: 성공 {len(results)} / 실패 {len(errors)}, 시간: {elapsed:.1f}s")

# 요약 CSV 저장
summary_csv = fld_dir / 'fld_to_h5_batch_summary.csv'
with summary_csv.open('w', newline='', encoding='utf-8') as csvf:
    fieldnames = ['variation','time_idx','fld','h5','n_points','fld_size','h5_size','removed']
    writer = csv.DictWriter(csvf, fieldnames=fieldnames)
    writer.writeheader()
    for r in results:
        writer.writerow({k: r.get(k, '') for k in fieldnames})
print(f"요약 저장: {summary_csv}")

In [ ]:
# ===== Export: 모든 Variation(30) × 모든 TimeStep(45) =====
from pathlib import Path
import csv
import time

# Ensure export_aedtplt.export_field is available
from export_aedtplt import export_field

# Output directory for .fld exports
output_dir = Path(m2d.project_path) / "FLD_Manual_Test"
output_dir.mkdir(parents=True, exist_ok=True)
print(f"Export output dir: {output_dir}")

# Validate parametric_table and time_steps
if 'parametric_table' not in globals():
    raise RuntimeError('parametric_table not found. Run data-prep cells first.')
if 'time_steps' not in globals():
    raise RuntimeError('time_steps not found. Run get_time_steps first.')

n_variations = len(parametric_table)
print(f"Total variations: {n_variations}")

# Export settings
FIELD_QUANTITY = field_quantity if 'field_quantity' in globals() else 'A'
TARGET_OBJECT = target_object if 'target_object' in globals() else 'Stator_Lamination_Primitive'
BATCH_SIZE = 10
SAVE_WAIT_TIME = 3
EXPORT_DELAY = 0.1

summary = []
start_all = time.perf_counter()
for i in range(n_variations):
    row = parametric_table.iloc[i]
    ipeak_val = row['IPeak']
    phase_val = row['PhaseAdvance']
    print(f"\n--- Exporting variation {i+1}/{n_variations}: IPeak={ipeak_val}, Phase={phase_val} ---")
    try:
        res = export_field(
            m2d_obj=m2d,
            output_dir=output_dir,
            setup_name=setup_name if 'setup_name' in globals() else 'Setup1',
            parametric_setup_name=parametric_setup_name if 'parametric_setup_name' in globals() else 'ParametricSetup1',
            field_quantity=FIELD_QUANTITY,
            target_object=TARGET_OBJECT,
            variation_index=i,
            batch_size=BATCH_SIZE,
            save_wait_time=SAVE_WAIT_TIME,
            export_delay=EXPORT_DELAY
        )
        res_row = {
            'variation_index': i,
            'IPeak': ipeak_val,
            'PhaseAdvance': phase_val,
            'success': res.get('success', False),
            'exported_count': res.get('exported_count', 0),
            'expected_count': res.get('expected_count', 0),
            'error': res.get('error', '')
        }
        summary.append(res_row)
        print(f"Result: success={res_row['success']}, exported={res_row['exported_count']}/{res_row['expected_count']}")
    except Exception as e:
        print(f"Export failed for variation {i}: {e}")
        import traceback
        traceback.print_exc()
        summary.append({'variation_index': i, 'IPeak': ipeak_val, 'PhaseAdvance': phase_val, 'success': False, 'exported_count': 0, 'expected_count': len(time_steps), 'error': str(e)})
    # small pause between variations
    time.sleep(0.5)

elapsed_all = time.perf_counter() - start_all
print(f"\nAll variations processed in {elapsed_all:.1f}s")

# Save summary CSV
summary_csv = output_dir / 'fld_export_all_variations_summary.csv'
with open(summary_csv, 'w', newline='', encoding='utf-8') as csvf:
    fieldnames = ['variation_index','IPeak','PhaseAdvance','success','exported_count','expected_count','error']
    writer = csv.DictWriter(csvf, fieldnames=fieldnames)
    writer.writeheader()
    for r in summary:
        writer.writerow(r)
print(f"Summary written to: {summary_csv}")

# Next step suggestion
print('\n✅ Export finished for all variations. Next: run the batch .fld→.h5 conversion cell to convert and remove .fld files.')

In [ ]:
# ===== Bulk .fld → .h5 변환 (폴더 전체) =====
from pathlib import Path
import time

fld_dir = Path(r"E:/KDH/e10/e10_DOE/e10_DOE.opd/AMOP/Design0121/FLD_Manual_Test")
if not fld_dir.exists():
    raise FileNotFoundError(f"폴더가 없습니다: {fld_dir}")

fld_files = sorted(fld_dir.glob('*.fld'))
print(f"변환 대상 .fld 파일 수: {len(fld_files)}")

# 변환 함수가 이미 정의되어 있으면 재사용, 없으면 정의
if 'convert_fld_to_h5' not in globals():
    import os, re, h5py, numpy as np
    FLOAT_RE = re.compile(r"[-+]?(?:\\d*\\.\\d+|\\d+\\.?(?:\\d*)?)(?:[eE][-+]?\\d+)?")
    def convert_fld_to_h5(fld_path, remove_fld=True, compression="gzip", compression_opts=6):
        fld_path = Path(fld_path)
        if not fld_path.exists():
            raise FileNotFoundError(f".fld file not found: {fld_path}")
        h5_path = fld_path.with_suffix('.h5')
        data_list = []
        with fld_path.open('r', encoding='utf-8', errors='ignore') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                parts = line.split()
                nums = None
                if len(parts) >= 4:
                    try:
                        cand = parts[:4]
                        nums = [float(x) for x in cand]
                    except Exception:
                        nums = None
                if nums is None:
                    found = FLOAT_RE.findall(line)
                    if len(found) >= 4:
                        try:
                            nums = [float(x) for x in found[:4]]
                        except Exception:
                            nums = None
                if nums is not None:
                    data_list.append(nums)
        if len(data_list) == 0:
            raise ValueError("No numeric data parsed from .fld file.")
        arr = np.asarray(data_list, dtype=np.float64)
        with h5py.File(h5_path, 'w') as hf:
            ds_kwargs = {}
            if compression:
                ds_kwargs['compression'] = compression
                ds_kwargs['compression_opts'] = compression_opts
            hf.create_dataset('field', data=arr, **ds_kwargs)
            hf.attrs['source_file'] = str(fld_path.name)
            hf.attrs['n_points'] = arr.shape[0]
        fld_size = fld_path.stat().st_size
        h5_size = h5_path.stat().st_size
        removed = False
        if remove_fld and h5_path.exists() and h5_size > 0:
            try:
                os.remove(fld_path)
                removed = True
            except Exception as e:
                print(f"⚠️ .fld 삭제 실패: {e}")
        return {'fld': str(fld_path), 'h5': str(h5_path), 'n_points': int(arr.shape[0]), 'fld_size': int(fld_size), 'h5_size': int(h5_size), 'removed': bool(removed)}

# 변환 루프
results = []
start = time.perf_counter()
for i, fpath in enumerate(fld_files, 1):
    try:
        print(f"[{i}/{len(fld_files)}] 변환 중: {fpath.name} ", end='', flush=True)
        res = convert_fld_to_h5(fpath, remove_fld=True, compression='gzip', compression_opts=9)
        results.append(res)
        print(f"✅ {res['n_points']} pts, .h5 {res['h5_size']/1024/1024:.2f} MB, removed={res['removed']}")
        # 잠깐 쉼
        time.sleep(0.1)
    except Exception as e:
        print(f"❌ 실패: {e}")
        import traceback
        traceback.print_exc()

elapsed = time.perf_counter() - start
print(f"\n전체 완료: {len(results)} 성공 / {len(fld_files)-len(results)} 실패, 시간: {elapsed:.1f}s")

# 결과 요약 csv 저장
import csv
summary_csv = fld_dir / 'fld_to_h5_summary.csv'
with open(summary_csv, 'w', newline='', encoding='utf-8') as csvf:
    writer = csv.DictWriter(csvf, fieldnames=['fld','h5','n_points','fld_size','h5_size','removed'])
    writer.writeheader()
    for r in results:
        writer.writerow(r)
print(f"요약 저장: {summary_csv}")

# 변환

In [ ]:
# ===== .fld → .h5 변환 + 원본 삭제 (안전 모드) =====
import os
import re
import h5py
import numpy as np
from pathlib import Path

FLOAT_RE = re.compile(r"[-+]?(?:\d*\.\d+|\d+\.?(?:\d*)?)(?:[eE][-+]?\d+)?")


def convert_fld_to_h5(fld_path, remove_fld=True, compression="gzip", compression_opts=6):
    """
    Convert a .fld file (text field export) to an HDF5 file and optionally remove the original .fld.

    Parameters
    ----------
    fld_path : str or Path
        Path to the .fld file to convert.
    remove_fld : bool
        If True, delete the .fld after successful conversion.
    compression : str or None
        h5py compression algorithm (e.g., 'gzip') or None.
    compression_opts : int
        Compression level for gzip (1-9).

    Returns
    -------
    dict
        { 'fld': str(fld_path), 'h5': str(h5_path), 'n_points': int, 'fld_size': int, 'h5_size': int, 'removed': bool }
    """
    fld_path = Path(fld_path)
    if not fld_path.exists():
        raise FileNotFoundError(f".fld file not found: {fld_path}")

    h5_path = fld_path.with_suffix('.h5')

    data_list = []
    with fld_path.open('r', encoding='utf-8', errors='ignore') as f:
        for ln, line in enumerate(f):
            line = line.strip()
            if not line:
                continue
            # Try simple split parse first (common case: x y z val)
            parts = line.split()
            nums = None
            if len(parts) >= 4:
                try:
                    # take first 4 numeric-looking tokens
                    cand = parts[:4]
                    nums = [float(x) for x in cand]
                except Exception:
                    nums = None
            if nums is None:
                # fallback: regex find floats
                found = FLOAT_RE.findall(line)
                if len(found) >= 4:
                    try:
                        nums = [float(x) for x in found[:4]]
                    except Exception:
                        nums = None
            if nums is not None:
                data_list.append(nums)
            # else: ignore non-data lines (headers/metadata)

    if len(data_list) == 0:
        raise ValueError("No numeric data parsed from .fld file. File format may be different.")

    arr = np.asarray(data_list, dtype=np.float64)

    # write to h5
    with h5py.File(h5_path, 'w') as hf:
        ds_kwargs = {}
        if compression:
            ds_kwargs['compression'] = compression
            ds_kwargs['compression_opts'] = compression_opts
        hf.create_dataset('field', data=arr, **ds_kwargs)
        # optional metadata
        hf.attrs['source_file'] = str(fld_path.name)
        hf.attrs['n_points'] = arr.shape[0]

    fld_size = fld_path.stat().st_size
    h5_size = h5_path.stat().st_size

    removed = False
    if remove_fld:
        # safety: only remove if h5 exists and is non-empty
        if h5_path.exists() and h5_size > 0:
            try:
                os.remove(fld_path)
                removed = True
            except Exception as e:
                print(f"⚠️ .fld 삭제 실패: {e}")
                removed = False

    return {
        'fld': str(fld_path),
        'h5': str(h5_path),
        'n_points': int(arr.shape[0]),
        'fld_size': int(fld_size),
        'h5_size': int(h5_size),
        'removed': bool(removed)
    }


# --- 사용자 지정 파일 경로 (테스트 대상) ---
test_fld = Path(r"E:/KDH/e10/e10_DOE/e10_DOE.opd/AMOP/Design0121/FLD_Manual_Test/Design0121_IPeak10A_Phase0deg_Time017.fld")

try:
    res = convert_fld_to_h5(test_fld, remove_fld=True, compression='gzip', compression_opts=9)
    print(f"변환 성공:\n - .fld: {res['fld']} ({res['fld_size']/1024/1024:.2f} MB)\n - .h5: {res['h5']} ({res['h5_size']/1024/1024:.2f} MB)\n - Points: {res['n_points']}\n - .fld removed: {res['removed']}")
except Exception as e:
    print(f"변환 실패: {e}")
    import traceback
    traceback.print_exc()

## backup

In [ ]:
import os
import numpy as np
import pandas as pd
import h5py
from io import StringIO

def convert_fld_to_h5(fld_path, h5_path=None, dataset_name="data"):
    """
    Maxwell .fld 텍스트 파일 → HDF5(.h5) 변환
    """
    if h5_path is None:
        base, _ = os.path.splitext(fld_path)
        h5_path = base + ".h5"

    # ----- 1) 파일 읽기 -----
    with open(fld_path, "r", encoding="utf-8", errors="replace") as f:
        lines = f.readlines()

    if len(lines) < 3:
        raise ValueError("파일 형식이 예상보다 짧음 (헤더 + NumElems + 데이터 필요)")

    header_line = lines[0].strip()
    numelems_line = lines[1].strip()

    # NumElems 파싱
    numelems = None
    parts = numelems_line.split()
    if len(parts) == 2 and parts[0].lower().startswith("numelems"):
        try:
            numelems = int(parts[1])
        except:
            pass

    # ----- 2) 데이터 부분 CSV 파싱 -----
    data_str = "".join(lines[2:])  # 3행부터 끝까지 데이터
    df = pd.read_csv(
        StringIO(data_str),
        delim_whitespace=True,
        header=None
    )

    data = df.to_numpy()

    # NumElems validation
    if numelems is not None:
        if data.shape[0] % numelems != 0:
            print(f"⚠️ 경고: 데이터 행수({data.shape[0]})가 NumElems({numelems}) 정수배가 아님")
        else:
            print(f"➡️ 데이터 구조: 요소당 {data.shape[0] // numelems} 행")

    # ----- 3) HDF5 저장 -----
    with h5py.File(h5_path, "w") as h5:
        h5.create_dataset(dataset_name, data=data)

        h5.attrs["source_file"] = os.path.basename(fld_path)
        h5.attrs["header_line"] = header_line
        h5.attrs["numelems_line"] = numelems_line
        if numelems is not None:
            h5.attrs["NumElems"] = numelems
        h5.attrs["rows"] = data.shape[0]
        h5.attrs["cols"] = data.shape[1]

    print(f"✅ 변환 완료: {h5_path}")
    return h5_path

In [ ]:
fld_path = r"E:\KDH\e10\e10_DOE\e10_DOE.opd\AMOP\Design0121\FLD_Manual_Test\Design0121_IPeak10A_Phase0deg_Time017.fld"
convert_fld_to_h5(fld_path)

In [ ]:
import h5py
import numpy as np
import pyvista as pv

# h5 읽기
h5_path = r"E:\KDH\e10\e10_DOE\e10_DOE.opd\AMOP\Design0121\FLD_Manual_Test\Design0121_IPeak10A_Phase0deg_Time017.h5"

with h5py.File(h5_path, "r") as f:
    data = f["data"][:]   # shape (45660, 6)

# 분리
xyz = data[:, :3]     # N x 3
vec = data[:, 3:6]    # N x 3

# PyVista point cloud 생성
cloud = pv.PolyData(xyz)
cloud["vec"] = vec  # vector field 등록

# 시각화
plotter = pv.Plotter()
plotter.add_mesh(cloud, scalars="vec", render_points_as_spheres=True)
plotter.add_arrows(cloud.points, cloud["vec"], mag=1.0)  # vector arrow 표시
plotter.show()